# KOS Pipeline Demo
### Automated Knowledge Base Population for NLB

This notebook walks through the pipeline step by step, showing what happens at each stage.

**Before running:** Make sure you've run `01_build_index.py` and `02_link_entities.py` first.

---

In [1]:
import sys
sys.path.insert(0, '..')

import json
import pickle
import pandas as pd

import config
import utils

print('Setup complete.')

Setup complete.


## 1. The Data We Start With

We have CNA articles already scraped, classified by event type, and NER'd.

In [2]:
articles = pd.read_csv('../data/cna_articles.csv', dtype=str).fillna('')

relevant = articles[
    (articles['Event_Type'] != 'not relevant') &
    (articles['Singapore_Context'].str.lower() == 'true')
]

print(f'Total articles scraped : {len(articles):,}')
print(f'Relevant articles      : {len(relevant):,}')
print(f'Relevance rate         : {len(relevant)/len(articles)*100:.1f}%')
print()
print('Top event types:')
print(relevant['Event_Type'].value_counts().head(10).to_string())

Total articles scraped : 2,970
Relevant articles      : 417
Relevance rate         : 14.0%

Top event types:
Event_Type
programme launched                        84
law enacted or amended                    42
building opened                           38
person received award                     35
person appointed or elected               33
building closed or ceased operations      31
organisation dissolved or closed          26
person died                               17
organisation restructured or rebranded    14
person resigned or retired                14


In [3]:
# Look at a sample article
sample = relevant.sample(1).iloc[0]
print(f"Title      : {sample['Title']}")
print(f"Event type : {sample['Event_Type']}")
print(f"Reasoning  : {sample['Reasoning']}")
print()

entities = json.loads(sample['extracted_entities'])
print('Extracted entities:')
for e in entities:
    print(f"  [{e['entity_label']}] {e['entity_name']} (confidence: {e['confidence']})")

Title      : Taste Orchard tenants shocked by abrupt notice to vacate after OG terminates Hao Mart's lease
Event type : building closed or ceased operations
Reasoning  : Taste Orchard (a specific named building/facility) is closing as tenants are being given notice to vacate after the lease termination, indicating the facility has ceased operations.

Extracted entities:
  [COUNTRY] SINGAPORE (confidence: high)
  [FACILITY] Taste Orchard (confidence: high)
  [ORGANISATION] Hao Mart (confidence: high)
  [ORGANISATION] OG (confidence: high)
  [ORGANISATION] CNA (confidence: high)
  [ORGANISATION] The Business Times (confidence: high)
  [ORGANISATION] BHC Chicken (confidence: high)
  [FACILITY] The Star Vista (confidence: high)
  [ORGANISATION] Facebook (confidence: high)
  [ORGANISATION] Eccellente (confidence: high)
  [ORGANISATION] Ikura Japanese (confidence: high)
  [LOCATION] Orchard Road (confidence: high)
  [ORGANISATION] Hao Corp (confidence: high)
  [PERSON] William Yang (confiden

## 2. The BM25 Index

We built a search index from all TTE alias surfaces.
Every name form, variant, and alias is indexed — the entity name from the article matches against all of them.

In [5]:
with open('../outputs/bm25_index.pkl', 'rb') as f:
    index = pickle.load(f)

print('BM25 Index loaded.')
print()
for vocab, vocab_index in index.items():
    n_entities = len(set(m['uid'] for m in vocab_index['metadata']))
    n_surfaces = len(vocab_index['surfaces'])
    print(f'  {vocab:<15} : {n_entities:,} entities, {n_surfaces:,} alias surfaces')

BM25 Index loaded.

  PERSON          : 22,258 entities, 40,300 alias surfaces
  ORGANISATION    : 11,775 entities, 20,712 alias surfaces
  FACILITY        : 3,418 entities, 6,824 alias surfaces
  LOCATION        : 8,998 entities, 14,429 alias surfaces
  EVENT           : 882 entities, 1,697 alias surfaces
  LEGAL_ACT       : 186 entities, 189 alias surfaces
  PROGRAMME       : 138 entities, 156 alias surfaces
  AWARD           : 368 entities, 510 alias surfaces


In [6]:
# Demo: search for an entity
query = 'Liu Thai Ker'
vocab = 'PERSON'

results = utils.search_index(index, query, vocab, top_k=5)

print(f"Query: '{query}' in {vocab}")
print()
if results:
    for r in results:
        print(f"  Score: {r['score']:.4f}  |  UID: {r['uid']}  |  {r['authorised_name']}")
        print(f"  Matched surface: '{r['matched_surface']}'  |  Record richness: {r['record_richness']} fields")
        print()
else:
    print('  No matches found.')

Query: 'Liu Thai Ker' in PERSON

  Score: 20.5784  |  UID: 18540823  |  Liu, Thai Ker
  Matched surface: 'Liu Thai Ker'  |  Record richness: 26 fields

  Score: 20.5784  |  UID: 18540823  |  Liu, Thai Ker
  Matched surface: 'Liu, Thai Ker'  |  Record richness: 26 fields

  Score: 20.5784  |  UID: 18552357  |  Liu Thai Ker
  Matched surface: 'Liu Thai Ker'  |  Record richness: 2 fields

  Score: 9.1701  |  UID: 18544105  |  Thai, Ha
  Matched surface: 'Thai Ha'  |  Record richness: 10 fields

  Score: 9.1701  |  UID: 18544105  |  Thai, Ha
  Matched surface: 'Thai, Ha'  |  Record richness: 10 fields



In [7]:
# Try with alias / variant name
query = 'URA'
vocab = 'ORGANISATION'

results = utils.search_index(index, query, vocab, top_k=3)
print(f"Query: '{query}' in {vocab}")
for r in results:
    print(f"  Score: {r['score']:.4f}  |  UID: {r['uid']}  |  {r['authorised_name']}")

Query: 'URA' in ORGANISATION
  Score: 13.6916  |  UID: 18487958  |  Urban Redevelopment Authority (Singapore)
  Score: 13.6916  |  UID: 18526475  |  URA


## 3. Entity Linking Results

After running `02_link_entities.py`, every entity from every relevant article has been searched against TTE.

In [8]:
linked = pd.read_csv('../outputs/linked_entities.csv', dtype=str).fillna('')

print(f'Total linked rows : {len(linked):,}')
print()
print('Match type distribution:')
print(linked['match_type'].value_counts().to_string())

Total linked rows : 1,915

Match type distribution:
match_type
high_match         1376
no_match            463
no_entity_found      62
low_match            14


In [9]:
# Show some high-confidence matches
high = linked[linked['match_type'] == 'high_match'].head(5)

print('Sample high-confidence matches:')
print()
for _, row in high.iterrows():
    print(f"  Article    : {row['article_title'][:60]}")
    print(f"  Event      : {row['event_type']}")
    print(f"  Extracted  : {row['entity_name']}")
    print(f"  → TTE match: {row['tte_authorised_name']}  (UID: {row['tte_uid']}, score: {row['match_score']})")
    print()

Sample high-confidence matches:

  Article    : DBS named world’s best private bank at global awards
  Event      : organisation received award
  Extracted  : DBS
  → TTE match: DBS Bank  (UID: 18545074, score: 12.024)

  Article    : DBS named world’s best private bank at global awards
  Event      : organisation received award
  Extracted  : DBS Digital Exchange
  → TTE match: DBS Bank  (UID: 18545074, score: 12.024)

  Article    : DBS named world’s best private bank at global awards
  Event      : organisation received award
  Extracted  : DBS Trustee
  → TTE match: DBS Bank  (UID: 18545074, score: 12.024)

  Article    : DBS named world’s best private bank at global awards
  Event      : organisation received award
  Extracted  : DBS Private Bank
  → TTE match: DBS Bank  (UID: 18545074, score: 17.7518)

  Article    : Urban Redevelopment Authority to get new CEO; next chief pla
  Event      : person appointed or elected
  Extracted  : Lim Eng Hwee
  → TTE match: Eng, Hwee Chu  (UI

## 4. Fact Extraction

For each high-confidence match, the LLM extracts the specific TTE field values from the article.

In [10]:
# Show what fields get extracted for each event type
print('Field mapping (event type → TTE fields to extract):')
print()
for event_type, fields in list(config.FIELD_MAPPING.items())[:8]:
    print(f"  {event_type}")
    print(f"    → {', '.join(fields)}")

Field mapping (event type → TTE fields to extract):

  person died
    → Death Year (yyyy), Death Month (mm), Death Day (dd)
  person appointed or elected
    → Affiliations(groupName), Occupation, Active Years, Description
  person resigned or retired
    → Active Years, Affiliations(groupName), Occupation, Description
  person changed affiliation or party
    → Affiliations(groupName)
  person received award
    → Awards
  person received honour or title
    → Achievements
  person got married
    → Spouse
  person renamed
    → Name


In [11]:
# If facts have been extracted, show sample
import os
if os.path.exists('../outputs/extracted_facts.csv'):
    facts = pd.read_csv('../outputs/extracted_facts.csv', dtype=str).fillna('')
    extracted = facts[facts['fact_status'] == 'extracted']
    print(f'Fact extraction results:')
    print(f'  Total processed   : {len(facts):,}')
    print(f'  Successfully extracted: {len(extracted):,}')
    print()

    for _, row in extracted.head(3).iterrows():
        fields = json.loads(row.get('extracted_fields', '{}'))
        non_null = {k: v['value'] for k, v in fields.items() if v and v.get('value')}
        if non_null:
            print(f"  {row['article_title'][:60]}")
            print(f"  Entity: {row['entity_name']} → {row['tte_authorised_name']}")
            print(f"  Extracted: {non_null}")
            print()
else:
    print('Run 03_extract_facts.py first to see results here.')

Run 03_extract_facts.py first to see results here.


## 5. Proposals

Everything is assembled into structured update proposals, ready for human review.

In [12]:
if os.path.exists('../outputs/proposals.json'):
    with open('../outputs/proposals.json') as f:
        proposals = json.load(f)

    priority_counts = {}
    for p in proposals:
        priority_counts[p['priority']] = priority_counts.get(p['priority'], 0) + 1

    print(f'Total proposals : {len(proposals):,}')
    for priority in ['HIGH', 'MEDIUM', 'FLAG']:
        print(f'  {priority:<8} : {priority_counts.get(priority, 0):,}')

    print()
    # Show a sample HIGH priority proposal
    high_proposals = [p for p in proposals if p['priority'] == 'HIGH' and p.get('proposed_changes')]
    if high_proposals:
        p = high_proposals[0]
        print('Sample HIGH priority proposal:')
        print(json.dumps(p, indent=2, ensure_ascii=False)[:1500])
else:
    print('Run 04_build_proposals.py first.')

Run 04_build_proposals.py first.


## 6. Export

After review, approved changes are exported as a CSV ready for TTE import.

In [13]:
if os.path.exists('../outputs/approved_changes.csv'):
    approved = pd.read_csv('../outputs/approved_changes.csv')
    print(f'Approved changes ready for TTE import: {len(approved)} field updates')
    print()
    print(approved[['tte_authorised_name', 'event_type', 'field', 'new_value', 'evidence']].head(10).to_string())
else:
    print('Run 05_review_queue.py and 06_export.py first.')

Run 05_review_queue.py and 06_export.py first.


---
## Pipeline Summary

```
2,970 articles scraped from CNA
    ↓  Call 1: LLM event classifier on title
421 relevant articles (14.2%)
    ↓  BM25 entity linking
??? high-confidence TTE matches
    ↓  Call 2: LLM fact extraction
??? structured field updates with evidence
    ↓  Human review
??? approved changes
    ↓  CSV export
Ready for TTE import
```

**Two LLM calls per article. No TTE writes without human approval. Full audit trail.**